# PerfectPetClub - Data Engineering ETL Pipeline
## Customer Churn Prediction & Analytics

### 1. Project Overview & Business Problem
**PerfectPetClub** is an online subscription service providing pet food and supplies. A key business challenge is **customer churn**—when customers disable the auto-renewal feature of their subscription (`auto_renew = 0`).

This ETL pipeline extracts data across relational and transactional sources, validates data quality, engineers predictive features for churn analysis, and exports the final curated dataset into an optimized Apache Parquet format.

### 2. Architecture: Extract - Transform - Load (ETL)
* **Extract:** Ingest customer profiles, subscriptions, sales, payments, and product reviews from SQLite (`PerfectPetClub.db`).
* **Transform:**
  1. *Data Quality Validation:* Run automated assertion checks (null handling, date order logic, identifier uniqueness, boundary checks).
  2. *Feature Engineering:* Construct the binary churn target (`is_churned`), subscription tenure, financial aggregations, and customer feedback metrics.
  3. *Data Ethics & GDPR:* Mask/drop personally identifiable information (PII) such as passwords and street addresses.
* **Load:** Write the clean customer-level feature dataset to `output/perfectpetclub_churn_features.parquet`.

In [1]:
# Import sys to manage Python's module search paths
import sys

# Import Path to work with file and folder locations
from pathlib import Path

# Add the src folder to Python's module search path
sys.path.insert(0, str(Path("../src").resolve()))

# Import the extraction function from crm.py
from crm import extract_crm

# Call the function to extract our CRM dataset
crm_df = extract_crm("../data/PerfectPetClub_CRM_report.csv")

# Display the first five rows to confirm extraction worked
display(crm_df.head())

.DS_Store
PerfectPetClub_CRM_report.csv
PerfectPetClub_server_log.csv


In [ ]:

# Test: Can our validation detect an invalid date?

test_data = raw_customers.copy()

# Deliberately insert an invalid subscription date
test_data.loc[0, 'sub_end_date'] = 'invalid-date'

try:
    validate_customer_data(test_data)
    print("TEST FAILED: Invalid date was accepted.")

except AssertionError as e:
    print("TEST PASSED: Invalid date was detected.")
    print(e)


## Stage 3: Transform — Feature Engineering & Integration
In this stage, we construct the predictive features for churn analysis:
* `is_churned`: **Target variable** (`1` if `auto_renew == 0` else `0`).
* `subscription_tenure_days`: Length of customer relationship in days.
* `avg_order_value_gbp`: Average transaction size.
* `one_star_reviews_count`: Dissatisfaction indicator.
* `postcode_district`: Area code for geographic segmentation.

In [ ]:
# 1. Transform customer features
clean_customers = transform_customer_features(raw_customers)

# 2. Merge all features on customer_id (1-to-1 relationship)
master_features = clean_customers.merge(raw_sales_reviews, on='customer_id', how='left')

# 3. Clean up PII for GDPR compliance (drop name, full postcode, and raw subscription IDs)
final_dataset = master_features.drop(columns=['customer_name', 'postcode', 'subscription_id'])

print(f'Master feature dataset shape: {final_dataset.shape}')
final_dataset.head()

### Feature Engineering Summary & Churn Analysis
Quick validation of churn distribution and key drivers:

In [ ]:
# Churn rate by subscription tier
tier_summary = final_dataset.groupby('subscription_tier')['is_churned'].agg(
    total_customers='count',
    churn_rate='mean'
).reset_index()

tier_summary['churn_rate'] = (tier_summary['churn_rate'] * 100).round(1).astype(str) + '%'
print('--- Churn Rate by Subscription Tier ---')
print(tier_summary)

# Churned vs Retained spend & review comparison
print('\n--- Mean Metrics by Churn Status ---')
print(final_dataset.groupby('is_churned')[['total_spend_gbp', 'total_orders', 'avg_star_rating', 'one_star_reviews_count']].mean().round(2))

## Stage 4: Load — Export Curated Dataset to Apache Parquet
As specified in Task 2, the final transformed dataset must be exported as an Apache Parquet file.
Parquet provides columnar compression, preserves column data types, and is optimized for downstream analytical queries.

In [ ]:
# Create output directory
output_dir = Path('../output')
output_dir.mkdir(parents=True, exist_ok=True)

parquet_path = output_dir / 'perfectpetclub_churn_features.parquet'

# Export to Parquet
final_dataset.to_parquet(parquet_path, index=False, engine='pyarrow')

print(f'✓ Successfully loaded {len(final_dataset)} records into Parquet format:')
print(f'  Location: {parquet_path.resolve()}')
print(f'  File size: {parquet_path.stat().st_size:,} bytes')

In [ ]:
# Verification: Read back parquet file to guarantee integrity
df_verified = pd.read_parquet(parquet_path)
print(f'Parquet verification passed: {df_verified.shape[0]} rows, {df_verified.shape[1]} columns.')
print('\nDataset Columns:')
for col in df_verified.columns:
    print(f' - {col} ({df_verified[col].dtype})')

In [ ]:

# Final verification of feature engineering and Parquet output

# Check 1: Customer records have not been duplicated
assert final_dataset['customer_id'].is_unique, "Duplicate customers found"

# Check 2: Customer count remains unchanged
assert len(final_dataset) == len(raw_customers), "Customer count mismatch"

# Check 3: Churn calculation is correct
expected_churn = (final_dataset['auto_renew'] == 0).astype(int)
assert (final_dataset['is_churned'] == expected_churn).all(), "Incorrect churn values"

# Check 4: Subscription durations are not negative
assert (final_dataset['subscription_tenure_days'] >= 0).all(), "Negative duration found"

# Check 5: Parquet file contains the expected records and columns
assert df_verified.shape == final_dataset.shape, "Parquet dataset shape mismatch"
assert list(df_verified.columns) == list(final_dataset.columns), "Parquet columns mismatch"

print("PASS: Customer records are unique")
print("PASS: Customer count is correct")
print("PASS: Churn calculations are correct")
print("PASS: Subscription durations are valid")
print("PASS: Parquet structure is verified")
